# Electronic parts detector: fine-tune YOLO

Trains a model that finds electronic parts (LED, resistor, button, chip…) in a phone photo.

**Before you start:** `Runtime → Change runtime type → T4 GPU`.

Pipeline:
1. Download public labeled datasets from Roboflow Universe
2. Auto-label **your own photos** with YOLO-World (an open-vocabulary detector), then fix mistakes by hand
3. Merge everything into one class list
4. Fine-tune YOLO11
5. Test and export for the web app (ONNX / TF.js)

Training time on a free T4: roughly 20–60 minutes, depending on dataset size.

In [ ]:
!pip -q install ultralytics roboflow
import os, glob, shutil, random, re, yaml
from pathlib import Path
from ultralytics import YOLO
print("GPU:", __import__("torch").cuda.is_available())

## 1. Our class list

One list for the whole project. Every public dataset gets mapped onto it; classes we don't need are dropped.
Edit it if your kit has other parts.

**This list must match `shared/classes.json` in the app repo exactly** (same names, same order). The web app reads that file, and `npm run build` fails if this cell and the JSON differ. To change classes, edit both.


In [ ]:
# Keep in sync with shared/classes.json (checked by scripts/check-classes.mjs)
CLASSES = [
    "resistor", "capacitor", "led", "diode", "transistor", "ic",
    "button", "buzzer", "potentiometer", "ldr", "battery",
    "breadboard", "arduino", "jumper_wire", "servo", "dc_motor",
    "ultrasonic_sensor", "display",
]

# Keyword rules: a source class name is matched against these IN ORDER.
# Order matters: "photoresistor" must hit ldr before resistor, "light emitting diode" must hit led before diode.
RULES = [
    (["ldr", "photoresistor", "lightdependent", "photocell"], "ldr"),
    (["potentiometer", "pot", "trimmer", "variableresistor"], "potentiometer"),
    (["resistor"], "resistor"),
    (["led", "lightemitting"], "led"),
    (["diode", "zener"], "diode"),
    (["capacitor", "cap"], "capacitor"),
    (["transistor", "mosfet", "bjt"], "transistor"),
    (["ic", "integratedcircuit", "chip", "ne555", "555", "dip"], "ic"),
    (["button", "pushbutton", "switch", "tactile"], "button"),
    (["buzzer", "piezo", "speaker"], "buzzer"),
    (["battery", "batteryholder", "cell"], "battery"),
    (["breadboard"], "breadboard"),
    (["arduino", "uno", "nano", "esp32", "leonardo"], "arduino"),
    (["jumper", "wire", "cable"], "jumper_wire"),
    (["servo"], "servo"),
    (["motor"], "dc_motor"),
    (["ultrasonic", "hcsr04"], "ultrasonic_sensor"),
    (["display", "lcd", "oled", "7segment", "sevensegment"], "display"),
]

def to_canonical(name):
    low = name.lower()
    joined = re.sub(r"[^a-z0-9]", "", low)
    tokens = set(re.findall(r"[a-z0-9]+", low))
    tokens |= {t[:-1] for t in tokens if t.endswith("s") and len(t) > 2}   # "ics" -> "ic", "leds" -> "led"
    for keys, target in RULES:
        # short keys must be a whole word ("led", "ic"); long keys can appear anywhere ("jumperwires")
        if any(k in tokens or k == joined.rstrip("s") or (len(k) > 3 and k in joined) for k in keys):
            return target
    return None  # class gets dropped

# sanity check
for s in ["Resistor", "Light Emitting Diode", "Photoresistor", "IntegratedCircuit", "Ics",
          "Red LED", "Arduino Nano", "JumperWires", "electrolytic capacitor", "Ceramic_Cap",
          "NE555 timer", "Multimeter", "Placa de Dezvoltare UNO R3"]:
    print(f"{s:25s} -> {to_canonical(s)}")

## 2. Download public datasets (Roboflow Universe)

1. Make a free account at roboflow.com and copy your API key (Settings → API Keys).
2. For each dataset you want, open it on universe.roboflow.com → **Download Dataset** → format **YOLOv8** → "show download code". Copy the `workspace`, `project`, and `version` values into `SOURCES` below.

The two below are examples I found; check each one's images look like *your* use case before relying on it.
Datasets of parts **soldered on circuit boards** help the repair feature; **loose parts on a table** help the build feature.

In [ ]:
ROBOFLOW_API_KEY = ""   # <- paste your key here

SOURCES = [
    # (workspace, project, version)
    ("school-p0xt8", "electric-components-kkgsp", 1),   # loose kit parts, very small (~19 images)
    ("electronic-components-3jxxm", "electronics4", 6), # ~4000 images, check that the version number is current
]

from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
downloaded = []
for ws, proj, ver in SOURCES:
    try:
        ds = rf.workspace(ws).project(proj).version(ver).download("yolov8", location=f"/content/src/{proj}")
        downloaded.append(ds.location)
        print("OK ", proj, "->", ds.location)
    except Exception as e:
        print("FAILED", proj, e)

## 3. Your own photos, auto-labeled

This is the most important data: photos of **your** kit parts, on **your** tables, under **your** lighting.

1. Take 150–300 photos with phones: parts spread on a table, different backgrounds, angles, lighting, some messy piles.
2. Upload them to Google Drive in a folder, e.g. `MyDrive/parts_photos/`.
3. Run the cell below. YOLO-World draws boxes from the class names alone; no labeling needed yet.
4. **Then review**: upload `/content/autolabel` to Roboflow (it reads YOLO format), fix wrong boxes, export again as YOLOv8, and put it in `SOURCES` above. Fixing pre-drawn boxes is ~5x faster than drawing from scratch.

If you skip the review, the cell still adds the auto-labels to training; that works for a first test but the model will copy YOLO-World's mistakes.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
MY_PHOTOS = "/content/drive/MyDrive/parts_photos"   # <- your folder

from ultralytics import YOLOWorld
world = YOLOWorld("yolov8x-worldv2.pt")
# Descriptive prompts work better for an open-vocabulary model than short class names.
PROMPTS = {
    "resistor": "small resistor with colored bands",
    "capacitor": "capacitor",
    "led": "LED light emitting diode",
    "diode": "black diode",
    "transistor": "transistor with three legs",
    "ic": "black integrated circuit chip",
    "button": "push button",
    "buzzer": "black round buzzer",
    "potentiometer": "potentiometer knob",
    "ldr": "photoresistor",
    "battery": "battery",
    "breadboard": "white breadboard",
    "arduino": "arduino board",
    "jumper_wire": "jumper wire",
    "servo": "servo motor",
    "dc_motor": "small dc motor",
    "ultrasonic_sensor": "ultrasonic sensor with two round eyes",
    "display": "lcd display",
}
world.set_classes([PROMPTS[c] for c in CLASSES])

out = Path("/content/autolabel"); (out/"images").mkdir(parents=True, exist_ok=True); (out/"labels").mkdir(exist_ok=True)
photos = [p for p in Path(MY_PHOTOS).glob("*") if p.suffix.lower() in (".jpg", ".jpeg", ".png")]
print(len(photos), "photos")
for p in photos:
    r = world.predict(str(p), conf=0.15, verbose=False)[0]
    shutil.copy(p, out/"images"/p.name)
    with open(out/"labels"/(p.stem + ".txt"), "w") as f:
        for (x, y, w, h), c in zip(r.boxes.xywhn.tolist(), r.boxes.cls.tolist()):
            f.write(f"{int(c)} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n")
with open(out/"data.yaml", "w") as f:
    yaml.safe_dump({"names": CLASSES, "nc": len(CLASSES)}, f)
print("Auto-labels written to", out)

# look at a few results
for p in photos[:3]:
    world.predict(str(p), conf=0.15, verbose=False)[0].show()

## 4. Merge everything into one dataset

In [ ]:
MERGED = Path("/content/merged")
shutil.rmtree(MERGED, ignore_errors=True)
for split in ("train", "valid"):
    (MERGED/split/"images").mkdir(parents=True); (MERGED/split/"labels").mkdir(parents=True)

def add_source(root, own_class_list=False, valid_frac=0.15):
    root = Path(root)
    names = yaml.safe_load(open(root/"data.yaml"))["names"]
    if isinstance(names, dict): names = [names[k] for k in sorted(names)]
    remap = {i: (CLASSES.index(n) if own_class_list else
                 (CLASSES.index(to_canonical(n)) if to_canonical(n) else None))
             for i, n in enumerate(names)}
    dropped = [n for i, n in enumerate(names) if remap[i] is None]
    if dropped: print(f"  {root.name}: dropping classes {dropped}")
    imgs = [p for p in root.rglob("*") if p.suffix.lower() in (".jpg", ".jpeg", ".png") and "images" in p.parts]
    n = 0
    for img in imgs:
        lbl = Path(str(img.parent).replace("images", "labels")) / (img.stem + ".txt")
        if not lbl.exists(): continue
        lines = []
        for line in open(lbl).read().splitlines():
            parts = line.split()
            if len(parts) != 5: continue          # skip segmentation polygons
            new = remap.get(int(parts[0]))
            if new is not None: lines.append(" ".join([str(new)] + parts[1:]))
        if not lines: continue
        split = "valid" if ("valid" in img.parts or "test" in img.parts) else "train"
        if own_class_list and random.random() < valid_frac: split = "valid"
        name = f"{root.name}_{n}_{img.name}"
        shutil.copy(img, MERGED/split/"images"/name)
        open(MERGED/split/"labels"/(Path(name).stem + ".txt"), "w").write("\n".join(lines) + "\n")
        n += 1
    print(f"  {root.name}: {n} images added")
    return n

random.seed(0)
source_counts = {}
for loc in downloaded:
    source_counts[Path(loc).name] = add_source(loc)
own_photo_count = 0
if Path("/content/autolabel/data.yaml").exists():
    own_photo_count = add_source("/content/autolabel", own_class_list=True)

yaml.safe_dump({"path": str(MERGED), "train": "train/images", "val": "valid/images",
                "names": CLASSES, "nc": len(CLASSES)}, open(MERGED/"data.yaml", "w"))

# class balance: classes with very few boxes need more photos
from collections import Counter
cnt = Counter()
for f in (MERGED/"train"/"labels").glob("*.txt"):
    for line in open(f): cnt[CLASSES[int(line.split()[0])]] += 1
for c in CLASSES: print(f"{c:20s} {cnt[c]}")

## 5. Fine-tune

`yolo11s` is a good balance for phones. Use `yolo11n` if the web app needs to run on weak phones.

In [ ]:
model = YOLO("yolo11s.pt")   # pretrained on COCO, we fine-tune it on our parts
results = model.train(
    data=str(MERGED/"data.yaml"),
    epochs=80, imgsz=640, batch=16,
    patience=20,              # stop early if validation stops improving
    project="/content/runs", name="parts",
)

## 6. Check the results

mAP50 above ~0.7 is good for a demo. Look at the per-class table: weak classes need more of your own photos.

In [ ]:
best = YOLO("/content/runs/parts/weights/best.pt")
metrics = best.val()
from IPython.display import Image, display
display(Image("/content/runs/parts/confusion_matrix_normalized.png", width=700))

# try it on one of your own photos
test_img = photos[0] if photos else next((MERGED/"valid"/"images").glob("*"))
best.predict(str(test_img), conf=0.3)[0].show()

## 6b. Write the model card

The app's **Model card** page (for the judges) reads `model_card.json`. This cell fills it from the real training run and saves a few example predictions. Copy `model_card.json` to `data/model_card.json` in the app repo and the images to `public/models/examples/`.


In [ ]:
import json, datetime
EX = Path("/content/model_card_examples"); shutil.rmtree(EX, ignore_errors=True); EX.mkdir()
val_imgs = sorted((MERGED/"valid"/"images").glob("*"))[:4]
examples = []
for i, p in enumerate(val_imgs):
    r = best.predict(str(p), conf=0.3, verbose=False)[0]
    name = f"example_{i}.jpg"
    r.save(filename=str(EX/name))
    examples.append({"image": f"/models/examples/{name}", "caption": f"{len(r.boxes)} parts found"})

box = metrics.box
ap50_by_class = {int(c): float(v) for c, v in zip(box.ap_class_index, box.ap50)}
n_train = len(list((MERGED/"train"/"images").glob("*")))
n_val = len(list((MERGED/"valid"/"images").glob("*")))
card = {
    "status": "trained",
    "model_name": "parts-detector",
    "base_model": "yolo11s (COCO pre-trained)",
    "trained_at": datetime.date.today().isoformat(),
    "image_size": 640,
    "epochs": int(model.trainer.epoch + 1) if getattr(model, "trainer", None) else None,
    "data": {
        "public_datasets": [
            {"name": proj, "url": f"https://universe.roboflow.com/{ws}/{proj}", "images": source_counts.get(proj)}
            for ws, proj, ver in SOURCES
        ],
        "own_photos": own_photo_count,
        "train_images": n_train,
        "val_images": n_val,
    },
    "metrics": {"map50": float(box.map50), "map50_95": float(box.map), "precision": float(box.mp), "recall": float(box.mr)},
    "per_class": [{"name": c, "train_boxes": int(cnt[c]), "map50": ap50_by_class.get(i)} for i, c in enumerate(CLASSES)],
    "examples": examples,
}
json.dump(card, open("/content/model_card.json", "w"), indent=2)
print(json.dumps(card["metrics"], indent=2))


## 7. Export for the web app and save to Drive

- **ONNX** runs in the browser with `onnxruntime-web`, or on a Python server.
- **TF.js** runs in the browser with TensorFlow.js.

In [ ]:
best.export(format="onnx", imgsz=640, simplify=True)
best.export(format="tfjs", imgsz=640)

SAVE = Path("/content/drive/MyDrive/parts_model"); SAVE.mkdir(exist_ok=True)
for f in ["best.pt", "best.onnx"]:
    shutil.copy(f"/content/runs/parts/weights/{f}", SAVE)
shutil.copytree("/content/runs/parts/weights/best_web_model", SAVE/"best_web_model", dirs_exist_ok=True)
yaml.safe_dump({"names": CLASSES}, open(SAVE/"classes.yaml", "w"))
shutil.copy("/content/model_card.json", SAVE)
shutil.copytree(EX, SAVE/"examples", dirs_exist_ok=True)
print("Saved to", SAVE)
print("Next: copy best.onnx → public/models/best.onnx, model_card.json → data/model_card.json, examples/ → public/models/examples/")
